# Experiment: Temp Cross-Method Run for `gwas_additive_score_sig_n60`

Objective:
- Run only `gwas_additive_score_sig_n60`.
- Force `swap_size = 2` for both MCMC-IS and SAMC proposals.
- Save outputs to a fresh timestamped directory under `results/cross_method_notebook/`.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    CrossMethodStudyConfig,
    MCMCWorkflowConfig,
    SAMCWorkflowConfig,
    _mcmc_eval_count,
    _steps_per_chain,
    create_timestamped_run_dir,
    load_selected_scenarios,
    run_cross_method_study,
    save_cross_method_outputs,
)

pd.set_option("display.max_columns", 100)
project_root


## Configuration

This notebook mirrors the main cross-method study, but restricts execution to one scenario and fixes `swap_size = 2`.


In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "cross_method_notebook"

SCENARIO_KEYS_OVERRIDE = ["gwas_additive_score_sig_n60"]
RUN_TAG = "cross_method_gwas_sig_n60_swap2"

ESTIMATION_POINTS = (750_000, 1_000_000, 2_500_000, 5_000_000, 7_500_000, 10_000_000, 15_000_000) if not FAST_MODE else (50_000, 100_000, 200_000)
N_REPEATS = 10 if not FAST_MODE else 2
N_JOBS = 1
MIN_TAIL_STATES = 2
BASE_SEED = 12_345
SWAP_SIZE = 2
run_dir = None
scenario_dir = None

MCMC_LOCAL_SCAN_STRATEGY = "adaptive_q"
MCMC_LOCAL_SCAN_OBJECTIVE = "varhat_qmatch_soft"
MCMC_PRODUCTION_ESTIMATOR_VARIANT = "selected_scan_plus_production"
MCMC_SCAN_REFINE_TO_SCREEN_RATIO = 1.0
MCMC_SCAN_FINAL_TO_SCREEN_RATIO = 2.0
MCMC_SCAN_FINALIST_COUNT = 4
MCMC_LOCAL_SCAN_Q_MULTIPLIERS = (0.001, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.33, 0.5, 1.0)
MCMC_LOCAL_SCAN_COARSE_Q_MULTIPLIERS = (0.001, 0.01, 0.10, 0.5)

cross_cfg = CrossMethodStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=N_REPEATS,
    base_seed=BASE_SEED,
    iid_density_samples=150_000 if not FAST_MODE else 10_000,
    min_tail_states=MIN_TAIL_STATES,
    n_jobs=N_JOBS,
)
base_mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=25_000 if not FAST_MODE else 1_000,
    tune_steps=3_000 if not FAST_MODE else 1_000,
    local_scan_screen_total_steps=14_000 if not FAST_MODE else 1_000,
    local_scan_refine_total_steps=14_000 if not FAST_MODE else 1_000,
    local_scan_total_steps=32_000 if not FAST_MODE else 6_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    production_estimator_variant=MCMC_PRODUCTION_ESTIMATOR_VARIANT,
    proposal_size=SWAP_SIZE,
    local_scan_strategy=MCMC_LOCAL_SCAN_STRATEGY,
    local_scan_objective=MCMC_LOCAL_SCAN_OBJECTIVE,
    local_scan_q_multipliers=MCMC_LOCAL_SCAN_Q_MULTIPLIERS,
    local_scan_coarse_q_multipliers=MCMC_LOCAL_SCAN_COARSE_Q_MULTIPLIERS,
    local_scan_refine_top_k=2,
    local_scan_refine_radius=1,
    local_scan_refine_max_q_points=6,
    local_scan_finalist_count=MCMC_SCAN_FINALIST_COUNT,
    local_scan_swap_counts=(SWAP_SIZE,),
)
samc_cfg = SAMCWorkflowConfig(
    n_bins=100,
    t0=1_000.0,
    trace_every=200 if not FAST_MODE else 50,
    lambda_min_pilot=10_000 if not FAST_MODE else 500,
    proposal_size=SWAP_SIZE,
)


def target_scan_budget_from_p0(p0: float) -> int:
    p0 = float(p0)
    if 1e-6 <= p0 <= 1e-4:
        return 500_000
    if 1e-8 <= p0 < 1e-6:
        return 1_000_000
    return 1_000_000


def stage_eval_total(total_steps: int, *, n_candidates: int, n_chains: int) -> int:
    steps_per_chain = _steps_per_chain(int(total_steps), int(n_chains))
    return int(n_candidates) * _mcmc_eval_count(steps_per_chain, int(n_chains))


def adaptive_candidate_counts(cfg: MCMCWorkflowConfig) -> tuple[int, int]:
    coarse_count = len(tuple(cfg.local_scan_coarse_q_multipliers)) or len(tuple(cfg.local_scan_q_multipliers))
    refine_count = int(cfg.local_scan_refine_max_q_points) if str(cfg.local_scan_strategy) == "adaptive_q" else 0
    return int(coarse_count), int(refine_count)


def split_scan_budget_for_scenario(scenario, cfg: MCMCWorkflowConfig) -> dict:
    target_beta_selection_budget = int(target_scan_budget_from_p0(scenario.exact_p))
    scan_budget_ex_pilot = max(int(target_beta_selection_budget) - int(cfg.pilot_samples), 1)
    coarse_count, refine_count = adaptive_candidate_counts(cfg)
    finalist_count = int(cfg.local_scan_finalist_count)
    refine_ratio = float(MCMC_SCAN_REFINE_TO_SCREEN_RATIO) if str(cfg.local_scan_strategy) == "adaptive_q" else 0.0
    final_ratio = float(MCMC_SCAN_FINAL_TO_SCREEN_RATIO)
    budget_units = float(coarse_count) + float(refine_count) * refine_ratio + float(finalist_count) * final_ratio
    screen_total_steps = max(int(scan_budget_ex_pilot / max(budget_units, 1.0)), 1)
    refine_total_steps = (
        int(max(1, round(refine_ratio * screen_total_steps)))
        if str(cfg.local_scan_strategy) == "adaptive_q"
        else None
    )
    final_total_steps = int(max(1, round(final_ratio * screen_total_steps)))
    expected_screen_eval_total = stage_eval_total(
        screen_total_steps,
        n_candidates=coarse_count,
        n_chains=int(cfg.local_scan_screen_chains),
    )
    expected_refine_eval_total = (
        stage_eval_total(
            int(refine_total_steps),
            n_candidates=refine_count,
            n_chains=int(
                cfg.local_scan_refine_chains
                if cfg.local_scan_refine_chains is not None
                else cfg.local_scan_screen_chains
            ),
        )
        if refine_total_steps is not None and refine_count > 0
        else 0
    )
    expected_final_eval_total = stage_eval_total(
        final_total_steps,
        n_candidates=finalist_count,
        n_chains=int(cfg.local_scan_chains),
    )
    return {
        "target_beta_selection_budget": int(target_beta_selection_budget),
        "screen_total_steps": int(screen_total_steps),
        "refine_total_steps": int(refine_total_steps) if refine_total_steps is not None else None,
        "final_total_steps": int(final_total_steps),
        "expected_beta_selection_budget_upper": int(
            int(cfg.pilot_samples)
            + int(expected_screen_eval_total)
            + int(expected_refine_eval_total)
            + int(expected_final_eval_total)
        ),
    }


def mcmc_cfg_for_scenario(scenario):
    scan_budget = split_scan_budget_for_scenario(scenario, base_mcmc_cfg)
    return replace(
        base_mcmc_cfg,
        proposal_size=SWAP_SIZE,
        local_scan_swap_counts=(SWAP_SIZE,),
        local_scan_screen_total_steps=int(scan_budget["screen_total_steps"]),
        local_scan_refine_total_steps=(
            int(scan_budget["refine_total_steps"])
            if scan_budget["refine_total_steps"] is not None
            else None
        ),
        local_scan_total_steps=int(scan_budget["final_total_steps"]),
    )


print(json.dumps({
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "RUN_TAG": RUN_TAG,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "N_REPEATS": N_REPEATS,
    "N_JOBS": N_JOBS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
    "SWAP_SIZE": SWAP_SIZE,
    "MCMC_LOCAL_SCAN_STRATEGY": MCMC_LOCAL_SCAN_STRATEGY,
    "MCMC_LOCAL_SCAN_OBJECTIVE": MCMC_LOCAL_SCAN_OBJECTIVE,
    "MCMC_PRODUCTION_ESTIMATOR_VARIANT": MCMC_PRODUCTION_ESTIMATOR_VARIANT,
}, indent=2))


## Load the Scenario

This confirms the exact benchmark metadata before the expensive run starts.


In [ ]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None,
    min_tail_states=MIN_TAIL_STATES,
)

if len(scenarios) != 1:
    raise RuntimeError(f"Expected exactly one scenario, found {len(scenarios)}")

scenario = scenarios[0]
scenario_mcmc_cfg = mcmc_cfg_for_scenario(scenario)

display(pd.DataFrame([
    {
        "scenario": scenario.key,
        "exact_p": scenario.exact_p,
        "tail_hits": scenario.exact_tail_hits,
        "n_perm": scenario.exact_n_perm,
        "exact_method": scenario.exact_method,
        "family": scenario.portfolio.get("family"),
        "rarity_band": scenario.portfolio.get("rarity_band"),
        "difficulty": scenario.portfolio.get("expected_difficulty"),
    }
]))

print(json.dumps({
    "scenario": scenario.key,
    "description": scenario.description,
    "mcmc_proposal_size": scenario_mcmc_cfg.proposal_size,
    "mcmc_local_scan_swap_counts": scenario_mcmc_cfg.local_scan_swap_counts,
    "target_beta_selection_budget": target_scan_budget_from_p0(scenario.exact_p),
    "local_scan_screen_total_steps": scenario_mcmc_cfg.local_scan_screen_total_steps,
    "local_scan_refine_total_steps": scenario_mcmc_cfg.local_scan_refine_total_steps,
    "local_scan_final_total_steps": scenario_mcmc_cfg.local_scan_total_steps,
}, indent=2))


## Run the Study

Running this cell will create a fresh timestamped output directory, execute the full study, and save the article-facing plots.


In [ ]:
run_dir = create_timestamped_run_dir(OUTPUT_ROOT, RUN_TAG) if SAVE_OUTPUTS else None
study = run_cross_method_study(
    scenario,
    cross_cfg=cross_cfg,
    mcmc_cfg=scenario_mcmc_cfg,
    samc_cfg=samc_cfg,
)

summary_df = pd.DataFrame(study["summary"]).sort_values(["checkpoint", "method"])
display(summary_df[[
    "checkpoint",
    "method",
    "mean_estimate",
    "rmse",
    "mean_variance_estimate",
    "mean_eval_incl_tuning",
    "mean_q_tilt_tail_share",
    "mean_ess",
    "mean_zero_rate",
    "mean_samc_max_rel_freq_error",
]])

scenario_dir = None
if SAVE_OUTPUTS and run_dir is not None:
    scenario_dir = run_dir / scenario.key
    save_cross_method_outputs(
        scenario,
        study,
        output_dir=scenario_dir,
        cross_cfg=cross_cfg,
        mcmc_cfg=scenario_mcmc_cfg,
        samc_cfg=samc_cfg,
    )

print(json.dumps({
    "run_dir": str(run_dir) if run_dir is not None else None,
    "scenario_dir": str(scenario_dir) if scenario_dir is not None else None,
    "mcmc_beta_selection_budget": study["mcmc_beta_selection_budget"],
    "mcmc_reported_checkpoints": study.get("mcmc_reported_checkpoints", []),
    "beta_used": study["beta_workflow"].get("beta_used"),
}, indent=2))


## Review Saved Plots

This shows the two article-facing plots after the run completes.


In [ ]:
if scenario_dir is None:
    raise RuntimeError("No saved output directory found. Run the previous cell with SAVE_OUTPUTS=True first.")

display(Image(filename=str(scenario_dir / "cross_method_max_budget.png")))
display(Image(filename=str(scenario_dir / "cross_method_convergence.png")))
scenario_dir
